# Traffic Dataset

#### Imports

In [61]:
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

#### Load Data

In [71]:

trafic = pd.read_csv("data/traffic.csv")
trafic.head()

,date,0,1,2,3,4,5,6,7,8,...,421,422,423,424,425,426,427,428,429,OT
0,2016-07-01 02:00:00,0.0048,0.0146,0.0289,0.0142,0.0064,0.0232,0.0162,0.0242,0.0341,...,0.0191,0.0146,0.0174,0.0152,0.0333,0.0390,0.0295,0.0205,0.0176,0.0121
1,2016-07-01 03:00:00,0.0072,0.0148,0.0350,0.0174,0.0084,0.0240,0.0201,0.0338,0.0434,...,0.0285,0.0164,0.0174,0.0147,0.0347,0.0413,0.0324,0.0245,0.0216,0.0136
2,2016-07-01 04:00:00,0.0040,0.0101,0.0267,0.0124,0.0049,0.0170,0.0127,0.0255,0.0332,...,0.0256,0.0104,0.0138,0.0096,0.0317,0.0392,0.0200,0.0184,0.0121,0.0107
3,2016-07-01 05:00:00,0.0039,0.0060,0.0218,0.0090,0.0029,0.0118,0.0088,0.0163,0.0211,...,0.0122,0.0059,0.0123,0.0060,0.0284,0.0342,0.0120,0.0158,0.0073,0.0071
4,2016-07-01 06:00:00,0.0042,0.0055,0.0191,0.0082,0.0024,0.0095,0.0064,0.0087,0.0144,...,0.0076,0.0036,0.0109,0.0048,0.0269,0.0326,0.0092,0.0140,0.0044,0.0039


#### Define target and features

In [ ]:
TARGET = "OT"

FEATURES = [str(i) for i in range(431)]

X = trafic[FEATURES].values
y = trafic[TARGET].values.reshape(-1, 1)

In [64]:
print(X.shape)
print(y.shape)

(17544, 410)
(17544, 1)


#### Train, validation and test split

In [65]:
n = len(trafic)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X[:train_end]
X_val   = X[train_end:val_end]
X_test  = X[val_end:]

y_train = y[:train_end]
y_val   = y[train_end:val_end]
y_test  = y[val_end:]


print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 12280
Validation: 2632
Test: 2632


#### Scale the data

In [66]:
X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train = X_scaler.fit_transform(X_train)
X_val   = X_scaler.transform(X_val)
X_test  = X_scaler.transform(X_test)

y_train = y_scaler.fit_transform(y_train)
y_val   = y_scaler.transform(y_val)
y_test  = y_scaler.transform(y_test)


#### Create sequences

In [67]:
LOOKBACK = 24

def create_sequences(X, y, lookback):
    X_seq = []
    y_seq = []

    for i in range(lookback, len(X)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])

    return np.array(X_seq), np.array(y_seq)


X_train, y_train = create_sequences(X_train, y_train, LOOKBACK)
X_val, y_val = create_sequences(X_val, y_val, LOOKBACK)
X_test, y_test = create_sequences(X_test, y_test, LOOKBACK)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (12256, 24, 410)
y_train shape: (12256, 1)


# RNN

#### Build the RNN

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(
        shape=(LOOKBACK, len(FEATURES))
    ),

    tf.keras.layers.SimpleRNN(128),

    tf.keras.layers.Dense(32, activation="relu"),

    tf.keras.layers.Dense(1)
])

#### Compile

In [ ]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])


#### Train

In [ ]:
history = model.fit(
    X_train,
    y_train,

    validation_data=(X_val, y_val),

    epochs=50,
    batch_size=32,

    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ]
)

Epoch 1/50
383/383 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0044 - mae: 0.0400 - val_loss: 0.0015 - val_mae: 0.0274
Epoch 2/50
383/383 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0013 - mae: 0.0264 - val_loss: 0.0016 - val_mae: 0.0274
Epoch 3/50
383/383 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 9.5965e-04 - mae: 0.0228 - val_loss: 0.0027 - val_mae: 0.0444
Epoch 4/50
383/383 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 9.1118e-04 - mae: 0.0220 - val_loss: 0.0013 - val_mae: 0.0251
Epoch 5/50
383/383 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 8.8965e-04 - mae: 0.0218 - val_loss: 0.0012 - val_mae: 0.0245
Epoch 6/50
383/383 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 8.1796e-04 - mae: 0.0209 - val_loss: 0.0017 - val_mae: 0.0294
Epoch 7/50
383/383 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 6.9313e-04 - mae: 0.0186 - val_loss: 0.0023 - val_mae: 0.0389
Epoch 8/50
383/383 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 7.4029e-04 - mae: 0.0195 - val_loss: 0.0011 - val_mae: 0.0265
Epoch 9/50
383/383 ━━━━━

#### Evaluate

In [ ]:
test_loss, test_mae = model.evaluate(
    X_test,
    y_test
)

print("Test loss:", test_loss)
print("Test MAE:", test_mae)

82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0030 - mae: 0.0347 
Test loss: 0.0030279571656137705
Test MAE: 0.034678418189287186


#### Predictions

In [ ]:
predictions = model.predict(X_test)

predictions = y_scaler.inverse_transform(predictions)
actual = y_scaler.inverse_transform(y_test)

82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [ ]:
print("Prediction -> Actual")
for i in range(10):
  print(f"{predictions[i]}->{actual[i]}")

Prediction -> Actual
[0.04140465]->[0.0449]
[0.0454172]->[0.0467]
[0.0482792]->[0.0513]
[0.05537733]->[0.0552]
[0.05722922]->[0.0611]
[0.06256103]->[0.0668]
[0.06297167]->[0.0709]
[0.06208908]->[0.0712]
[0.06003116]->[0.0728]
[0.05596205]->[0.0637]
